Cellule 1 : Les importations et la configuration
Dans cette première cellule, on ramène tous les outils dont on a besoin, notamment TensorFlow pour construire le réseau de neurones et Scikit-Learn pour mélanger/séparer nos données.

In [1]:
import os
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import TensorBoard

# Configuration de base (doit correspondre à notre script de collecte)
DATA_PATH = os.path.join('dataset')
sequence_length = 30 # Nos 30 images par mouvement

# On détecte automatiquement les mots que tu as enregistrés dans le dossier 'dataset'
actions = np.array([name for name in os.listdir(DATA_PATH) if os.path.isdir(os.path.join(DATA_PATH, name))])
print(f"Mots détectés pour l'entraînement : {actions}")

# On crée un dictionnaire pour associer chaque mot à un numéro (ex: 'bonjour' -> 0, 'merci' -> 1)
label_map = {label:num for num, label in enumerate(actions)}
print(f"Mapping des labels : {label_map}")

ModuleNotFoundError: No module named 'sklearn'

Cellule 2 : Chargement et formatage des données
Ici, on va parcourir tous tes fichiers .npy, les charger en mémoire, et préparer les "étiquettes" (labels) pour que l'IA sache quelle vidéo correspond à quel mot.

In [ ]:
sequences, labels = [], []

for action in actions:
    # On compte combien de fichiers .npy (d'essais) on a pour cette action
    num_sequences = len(os.listdir(os.path.join(DATA_PATH, action)))
    
    for sequence in range(num_sequences):
        try:
            # On charge le fichier NumPy
            res = np.load(os.path.join(DATA_PATH, action, f"{sequence}.npy"))
            sequences.append(res)
            labels.append(label_map[action])
        except Exception as e:
            print(f"Erreur avec la séquence {sequence} du mot {action} : {e}")

# X = nos vidéos (les points), y = nos réponses (les mots)
X = np.array(sequences)
y = to_categorical(labels).astype(int) # Transforme [0, 1] en format binaire compréhensible par l'IA

# On sépare les données : 95% pour apprendre, 5% pour tester si elle a bien appris
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.05)

print(f"Forme des données d'entraînement (Vidéos, Frames, Points) : {X_train.shape}")

Cellule 3 : Construction de l'architecture LSTM
C'est ici qu'on construit notre réseau de neurones récurrent. Les couches LSTM vont analyser l'évolution des 126 points au fil des 30 frames pour comprendre la dynamique de ton geste.

In [ ]:
model = Sequential()

# Première couche LSTM : elle lit les 30 frames de 126 points
model.add(LSTM(64, return_sequences=True, activation='relu', input_shape=(sequence_length, 126)))
model.add(Dropout(0.2)) # On désactive 20% des neurones au hasard pour éviter que l'IA apprenne "par coeur" (overfitting)

# Deuxième couche LSTM
model.add(LSTM(128, return_sequences=True, activation='relu'))
model.add(Dropout(0.2))

# Troisième couche LSTM (return_sequences=False car c'est la dernière étape d'analyse temporelle)
model.add(LSTM(64, return_sequences=False, activation='relu'))

# Couches de décision classiques
model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))

# Couche finale : autant de neurones que de mots à deviner
model.add(Dense(actions.shape[0], activation='softmax'))

# Compilation du modèle
model.compile(optimizer='Adam', loss='categorical_crossentropy', metrics=['categorical_accuracy'])
model.summary()

Cellule 4 : L'entraînement !
On lance l'apprentissage. epochs=100 signifie que l'IA va repasser 100 fois sur tes données pour affiner ses connexions. Ça devrait aller très vite sur ton Mac (quelques secondes à peine).

(Tu vas voir les lignes défiler. L'objectif est que la valeur de loss baisse et que categorical_accuracy se rapproche de 1.0).

In [ ]:
# On lance l'entraînement
history = model.fit(X_train, y_train, epochs=100, validation_data=(X_test, y_test))

Cellule 5 : Sauvegarde du modèle
Une fois que l'IA est entraînée et performante, on sauvegarde son "cerveau" dans un fichier. C'est ce fichier que ton API Flask utilisera plus tard !

In [ ]:
# Sauvegarde au format standard Keras
model.save('modele_lsf.keras')
print("Modèle sauvegardé avec succès sous le nom 'modele_lsf.keras' !")

# Petit test rapide sur une donnée de test pour voir si ça marche
res = model.predict(X_test)
prediction_index = np.argmax(res[0])
vrai_index = np.argmax(y_test[0])

print(f"L'IA a prédit : {actions[prediction_index]}")
print(f"La vraie réponse était : {actions[vrai_index]}")